# Đề xuất model churn cho stakeholder: có nên làm, làm cỡ nào, đo thế nào

**Bối cảnh.** COO của nhà mạng nhận được hai đề xuất: đội Customer Care muốn **giữ quy tắc** đang dùng (gọi khách
month-to-month gắn bó tối đa 12 tháng), đội Data muốn **thay bằng model churn**. COO hỏi bạn, DA trung lập: *"Model có
đáng tiền không? Nếu có, gọi bao nhiêu khách mỗi kỳ? Rủi ro là gì, và làm sao biết nó có tác dụng thật?"* Bạn viết một
đề xuất một trang (`starter/PROPOSAL.md`) cho COO, người không đọc code.

Dữ liệu: bộ `telco-churn` (IBM, 7 043 khách, 26.5% rời bỏ). Model ứng viên là logistic regression đã dựng sẵn bên dưới
(cùng 10 đặc trưng của lý thuyết phần 3). Các con số tiền (chi phí một lần liên hệ, giá trị một khách giữ được, tỉ lệ
giữ được khi gọi) là **ví dụ giả định**: dữ liệu không có chúng, và một phần việc của bạn là nói rõ kết luận phụ thuộc
vào giả định nào.

**Phần được chấm tự động**: hàm `campaign_table` (cell bên dưới) và khung `PROPOSAL.md`. Phần còn lại là phân tích tự do,
chấm theo rubric trong lời giải.

Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/advanced/01-churn-model-proposal -q`.

In [ ]:
# explore: dữ liệu, đặc trưng và model ứng viên (đã dựng sẵn)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

raw = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
X = pd.DataFrame({
    "contract_one_year": (raw.Contract == "One year").astype(int),
    "contract_two_year": (raw.Contract == "Two year").astype(int),
    "tenure_years": raw.tenure / 12,
    "fiber_optic": (raw.InternetService == "Fiber optic").astype(int),
    "no_internet": (raw.InternetService == "No").astype(int),
    "electronic_check": (raw.PaymentMethod == "Electronic check").astype(int),
    "monthly_charges_10": raw.MonthlyCharges / 10,
    "senior": raw.SeniorCitizen,
    "tech_support": (raw.TechSupport == "Yes").astype(int),
    "paperless": (raw.PaperlessBilling == "Yes").astype(int),
})
y = (raw.Churn == "Yes").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
logit = sm.Logit(y_train, sm.add_constant(X_train)).fit(disp=0)
score_test = logit.predict(sm.add_constant(X_test))
rule_test = ((X_test.contract_one_year == 0) & (X_test.contract_two_year == 0) & (X_test.tenure_years <= 1)).astype(int)
print(f"test: {len(y_test):,} khách, {y_test.sum()} rời bỏ; quy tắc gắn cờ {rule_test.sum()} khách")

In [ ]:
import numpy as np
import pandas as pd

## Hàm được chấm: bảng chiến dịch theo ngân sách

`campaign_table(y_true, score, budgets, contact_cost, value_per_save, save_rate) -> DataFrame`. Với mỗi số khách liên hệ
`k` trong `budgets` (giữ thứ tự), chọn `k` khách điểm cao nhất (sắp giảm dần **ổn định**: hoà điểm giữ thứ tự đầu vào).
Index tên `contacted` (= `k`), cột đúng thứ tự:

| Cột | Cách tính |
| --- | --- |
| `churners_reached` | số khách rời bỏ trong `k` khách được liên hệ (`int`) |
| `capture` | `churners_reached` / tổng số khách rời bỏ |
| `precision` | `churners_reached / k` (`0.0` khi `k = 0`) |
| `expected_saves` | `churners_reached * save_rate` |
| `cost` | `k * contact_cost` |
| `expected_value` | `expected_saves * value_per_save` |
| `net` | `expected_value - cost` |
| `random_net` | `net` kỳ vọng nếu chọn `k` khách ngẫu nhiên: `k * tỉ lệ rời chung * save_rate * value_per_save - cost` |
| `net_vs_random` | `net - random_net` |

`ValueError` khi: `y_true` và `score` khác độ dài; một `k` không phải số nguyên trong `[0, n]`; `save_rate` ngoài
`[0, 1]`; `contact_cost` hoặc `value_per_save` âm.

Lưu ý mô hình tiền ở đây giả định chỉ khách **sắp rời** mới tạo giá trị khi được gọi. Khách không định rời được gọi chỉ tốn
chi phí (và có thể tốn hơn: ưu đãi cho người vốn không cần). Đó là một giả định cần nói trong đề xuất.

In [ ]:
def campaign_table(y_true, score, budgets, contact_cost: float, value_per_save: float,
                   save_rate: float) -> pd.DataFrame:
    """Hiệu quả và tiền kỳ vọng khi liên hệ k khách điểm cao nhất, so với chọn ngẫu nhiên."""
    # TODO: kiểm đầu vào (ValueError)
    # TODO: order = np.argsort(-score, kind="stable"); reached cộng dồn theo thứ tự đó
    # TODO: một dòng cho mỗi k trong budgets
    raise NotImplementedError

In [ ]:
# explore: giả định ban đầu (ví dụ giả định, ghi rõ trong đề xuất)
CONTACT_COST = 15.0     # USD: cuộc gọi + ưu đãi nhỏ
VALUE_PER_SAVE = 600.0  # USD: khoảng 65 USD cước tháng trung bình × 9 tháng ở lại thêm
SAVE_RATE = 0.25        # 1/4 khách sắp rời được gọi sẽ ở lại

budgets = [100, 200, 300, 400, 476, 600, 800, 1000, 1200, len(y_test)]
pd.set_option("display.width", 180)
print(campaign_table(y_test, score_test, budgets, CONTACT_COST, VALUE_PER_SAVE, SAVE_RATE).round(3))

In [ ]:
# explore: quy tắc đang dùng, cùng giả định tiền
k_rule = int(rule_test.sum())
reached_rule = int(y_test[rule_test == 1].sum())
net_rule = reached_rule * SAVE_RATE * VALUE_PER_SAVE - k_rule * CONTACT_COST
model_same_k = campaign_table(y_test, score_test, [k_rule], CONTACT_COST, VALUE_PER_SAVE, SAVE_RATE).iloc[0]
print(f"quy tắc: liên hệ {k_rule}, bắt {reached_rule} khách rời, net {net_rule:,.0f} USD trên tập test")
print(f"model cùng {k_rule} khách: bắt {model_same_k.churners_reached:.0f}, net {model_same_k.net:,.0f} USD")

## Phân tích tự do (gợi ý các bước)

1. **Ngân sách tối ưu**: với giả định ban đầu, `net` lớn nhất ở cỡ nào? Vẽ `net` theo `contacted` cho model, quy tắc và
   ngẫu nhiên.
2. **Độ nhạy**: kết luận có đổi không khi `SAVE_RATE` là 0.10 hoặc 0.40, `VALUE_PER_SAVE` là 300 hoặc 900? Tìm `save_rate`
   hoà vốn ở ngân sách bạn chọn.
3. **Từ tập test tới cả công ty**: tập test có 1 761 khách (25%). Quy đổi về toàn bộ khách mỗi kỳ thế nào, và con số đó
   chắc chắn đến đâu?
4. **Model tốt đều không?** Kiểm precision của top-k theo `Contract` hoặc `InternetService`. Model có chỉ đang tìm lại
   quy tắc không?
5. **Pilot**: thiết kế cách đo tác động thật: nhóm đối chứng ngẫu nhiên trong số khách được model chọn, metric chính,
   thời gian, cỡ mẫu sơ bộ.

Rồi viết `starter/PROPOSAL.md` theo khung có sẵn.

In [ ]:
# explore: chỗ làm việc của bạn